# Step 1: Reading & Validating the Data

**Project:** Spotify Music Recommendation System
**Phase:** 1 of 4 — Reading Data

## What this notebook does

Before we can analyze anything or build a recommendation system, we need to
know exactly what data we're working with: how much of it there is, what
each column means, and whether it's trustworthy. This notebook loads our two
datasets, explains what's in them, and checks them for common data-quality
problems (missing values, duplicates, malformed entries) so that every step
after this one can be built on solid ground.

## Why two datasets?

This project actually uses **two separate datasets**, each for a different
purpose:

| Dataset | What it contains | What we'll use it for |
|---|---|---|
| **Audio-features dataset** (1921–2020, ~170k tracks) | Numerical audio characteristics of each song — how danceable it is, how energetic, how fast, etc. | Clustering songs by sound (Step 3) and building the recommendation engine (Step 4) |
| **Genre & metadata dataset** (2009–2025, ~8.8k tracks) | Genre tags, artist popularity, follower counts — but *no* audio characteristics | Genre-based exploration and word clouds (Step 2) |

**Why not just use one dataset for everything?** Spotify changed its public
API rules in November 2024 and stopped allowing new developers to access
audio characteristics (danceability, tempo, etc.) for tracks. This means any
dataset collected *after* that date — like our more recent genre dataset —
simply can't contain that information anymore, no matter how it was built.
Our older dataset was collected before that change, so it still has the full
audio picture, just not for anything released after 2020. Splitting the work
between the two datasets lets us get the best of both: real audio-based
modeling *and* an up-to-date view of genres.


## 1.1 Setup

We start by importing the libraries we'll need. `pandas` is the core tool
for loading and inspecting tabular data — think of it as a way to work with
spreadsheet-like data programmatically.

In [1]:
import pandas as pd

# Show all columns when we print a table, instead of truncating
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1.2 Loading the audio-features dataset

This is our **primary dataset** — it will act as the backbone for the
clustering and recommendation work later in this project. Each row is one
song, and each column is either an identifying detail (name, artist, year)
or a numerical measurement of the song's sound.

In [6]:
audio_df = pd.read_csv(r"../data/raw/spotify_tracks.csv")

print(f"Loaded {audio_df.shape[0]:,} rows and {audio_df.shape[1]} columns")
audio_df.head(3)


Loaded 169,909 rows and 19 columns


,id,name,artists,duration_ms,release_date,year,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,mode,key,popularity,explicit
0,6KbQ3uYMLKb5jDxLF7wYDD,Singende Bataillone 1. Teil,['Carl Woitschach'],158648,1928,1928,0.995,0.708,0.1950,0.563,0.1510,-12.428,0.0506,118.469,0.7790,1,10,0,0
1,6KuQTIu1KoTTkLXKrwlLPV,"Fantasiestücke, Op. 111: Più tosto lento","['Robert Schumann', 'Vladimir Horowitz']",282133,1928,1928,0.994,0.379,0.0135,0.901,0.0763,-28.454,0.0462,83.972,0.0767,1,8,0,0
2,6L63VW0PibdM1HDSBoqnoM,Chapter 1.18 - Zamek kaniowski,['Seweryn Goszczyński'],104300,1928,1928,0.604,0.749,0.2200,0.000,0.1190,-19.924,0.9290,107.177,0.8800,0,5,0,0


### What do these columns mean?

Most of these column names aren't self-explanatory unless you already know
how Spotify measures music, so here's a plain-language guide:

| Column | What it actually means |
|---|---|
| `danceability` | How suitable the track is for dancing, from 0 (least) to 1 (most), based on tempo, rhythm stability, and beat strength |
| `energy` | A 0–1 measure of intensity — high energy feels fast, loud, and noisy (e.g. death metal); low energy feels calm (e.g. a Bach prelude) |
| `acousticness` | A 0–1 confidence score for whether the track is acoustic (non-electronic) |
| `instrumentalness` | How likely the track has no vocals — closer to 1 means "probably instrumental" |
| `liveness` | Detects the presence of a live audience in the recording — higher values suggest a live performance |
| `loudness` | Overall loudness in decibels (dB), averaged across the track — typically ranges from about -60 to 0 |
| `speechiness` | How much the track resembles spoken word (e.g. a podcast or rap verse) versus melodic singing |
| `tempo` | The track's speed, measured in beats per minute (BPM) |
| `valence` | How musically positive/happy the track sounds, from 0 (sad, angry) to 1 (happy, cheerful) |
| `mode` | Whether the track is in a major (1) or minor (0) key |
| `key` | The musical key the track is in, using standard pitch class notation (0 = C, 1 = C♯, etc.) |
| `popularity` | A 0–100 score reflecting how popular the track was on Spotify at the time of collection |

These twelve numbers are what let us mathematically compare how "similar"
two songs sound — which is the whole foundation of Steps 3 and 4.

## 1.3 Checking the audio-features dataset for problems

Real-world data is rarely perfect. Before trusting this dataset, we check
for three common issues:

1. **Missing values** — blank or null entries that could break calculations later
2. **Duplicate rows** — the exact same song appearing more than once
3. **Values outside expected ranges** — e.g. a `danceability` score of 5 would be
   suspicious, since it should always fall between 0 and 1

If we find problems, we'll need to decide whether to clean, drop, or
otherwise handle the affected rows before moving to EDA.

In [8]:
print("Missing values per column:")
missing = audio_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None found — every column is fully populated.")

print()
print(f"Duplicate rows: {audio_df.duplicated().sum()}")
print(f"Duplicate track IDs: {audio_df['id'].duplicated().sum()}")


Missing values per column:
None found — every column is fully populated.

Duplicate rows: 0
Duplicate track IDs: 0


In [9]:
# Sanity-check that the 0-1 bounded features are actually within range
zero_to_one_features = [
    "acousticness", "danceability", "energy",
    "instrumentalness", "liveness", "speechiness", "valence"
]
range_check = audio_df[zero_to_one_features].agg(["min", "max"])
print("Min/max for features that should fall between 0 and 1:")
range_check


Min/max for features that should fall between 0 and 1:


,acousticness,danceability,energy,instrumentalness,liveness,speechiness,valence
min,0.000,0.000,0.0,0.0,0.0,0.000,0.0
max,0.996,0.988,1.0,1.0,1.0,0.969,1.0


### What this tells us

The dataset has **no missing values and no duplicate rows or track IDs**,
and every feature that should be bounded between 0 and 1 actually is. This
is a clean dataset — we won't need to spend time on data cleaning before
moving to the exploration phase.

In [10]:
print(f"Year coverage: {audio_df['year'].min()} to {audio_df['year'].max()}")


Year coverage: 1921 to 2020


## 1.4 Loading the genre & metadata dataset

Our second dataset is more recent (up to 2025) and gives us genre tags and
popularity metadata that the audio-features dataset can't — but as
explained above, it has no danceability/energy/tempo-style features. We'll
use this purely for genre-based exploration in Step 2.

In [11]:
genre_df = pd.read_csv("../data/raw/spotify_genre_metadata.csv")

print(f"Loaded {genre_df.shape[0]:,} rows and {genre_df.shape[1]} columns")
genre_df.head(3)


Loaded 8,778 rows and 15 columns


,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,['pop'],325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album


### What do these columns mean?

| Column | What it actually means |
|---|---|
| `track_popularity` | 0–100 popularity score for the individual track |
| `artist_popularity` | 0–100 popularity score for the artist overall |
| `artist_followers` | Number of Spotify followers the artist has |
| `artist_genres` | A list of genre tags associated with the artist (e.g. `['pop', 'dance pop']`) |
| `album_type` | Whether the release is an `album`, `single`, or `compilation` |
| `explicit` | Whether the track is flagged as containing explicit content |

Note this dataset is organized by **artist-level genre tags**, not
per-track genres — Spotify doesn't tag individual songs with genres, only
artists. That's a small but important nuance for how we build genre
word clouds in Step 2.

## 1.5 Checking the genre dataset for problems

Same checks as before: missing values, duplicates, and a look at whether
the data looks internally consistent (e.g. do repeated artists have
consistent follower counts?).

In [12]:
print("Missing values per column:")
missing = genre_df.isnull().sum()
print(missing[missing > 0])

print()
print(f"Duplicate rows: {genre_df.duplicated().sum()}")
print(f"Duplicate track IDs: {genre_df['track_id'].duplicated().sum()}")
print(f"Unique artists: {genre_df['artist_name'].nunique():,}")


Missing values per column:
track_name           2
artist_name          4
artist_popularity    4
artist_followers     4
artist_genres        4
album_name           2
dtype: int64

Duplicate rows: 0
Duplicate track IDs: 0
Unique artists: 2,548


### What this tells us

There are a small number of missing values — a handful of tracks are
missing artist metadata (4 rows) or a track/album name (2 rows). Given this
is only a few rows out of nearly 8,800, we can safely drop them during
cleanup in Step 2 rather than trying to fill them in. There are no
duplicate rows or track IDs, which means every row represents a genuinely
distinct song.

In [13]:
# Confirm consistency: the same artist should always show the same follower count
consistency_check = genre_df.groupby("artist_name")["artist_followers"].nunique()
inconsistent_artists = consistency_check[consistency_check > 1]
print(f"Artists with inconsistent follower counts across their tracks: {len(inconsistent_artists)}")


Artists with inconsistent follower counts across their tracks: 796


Roughly 800 artists show more than one follower count across their
tracks. At first glance that might look like a data-quality problem, but
it's actually a believable sign of *real* data: follower counts change
over time, and if this dataset was assembled by querying the Spotify API
across multiple sessions (rather than one single snapshot), the same
artist could reasonably be recorded with slightly different follower
counts depending on when each track was fetched. This is the kind of
small inconsistency that's very hard to fake convincingly, and unlikely to
show up in a purely synthetic dataset, where values are usually generated
independently per row with no real-world timing behind them.

We'll keep this in mind for Step 2: for any analysis involving artist
follower counts, we should use the *latest* or *most frequent* value per
artist rather than assuming there's a single canonical number.

## 1.6 How these two datasets fit together

It's worth being clear about something: these two datasets **won't be
merged row-by-row**. They cover different, mostly non-overlapping sets of
tracks (the audio dataset stops at 2020; the genre dataset starts more
heavily post-2009 and includes recent releases), and they were collected
independently. Instead, they'll be used **side by side**, each for the part
of the project they're actually suited to:

- The **audio-features dataset** drives Steps 3–4 (clustering and
  recommendations), since that's the only one with the numerical sound
  characteristics that similarity calculations depend on.
- The **genre dataset** enriches Step 2 (exploratory analysis) with
  genre-based insights — like which genres are most common, and genre word
  clouds — that the older dataset can't provide on its own.

Just to get a feel for how much these two worlds overlap, here's a quick
check of how many artists appear in both datasets.

In [14]:
audio_artists = set(audio_df["artists"].str.strip("[]'\"").str.split(",").str[0].str.strip())
genre_artists = set(genre_df["artist_name"].dropna())

shared = audio_artists & genre_artists
print(f"Artists in audio-features dataset: {len(audio_artists):,}")
print(f"Artists in genre dataset: {len(genre_artists):,}")
print(f"Artists appearing in both: {len(shared):,}")


Artists in audio-features dataset: 22,752
Artists in genre dataset: 2,548
Artists appearing in both: 1,214


This overlap confirms both datasets are drawing from the same real
universe of artists, even though the specific tracks and time periods
differ — a reassuring consistency check.

## Summary

- **Audio-features dataset**: 169,909 tracks (1921–2020), 19 columns, no
  missing values, no duplicates. Will be the backbone for clustering
  (Step 3) and the recommendation engine (Step 4).
- **Genre & metadata dataset**: 8,778 tracks (2009–2025), 15 columns, a
  handful of missing values (to be dropped), no duplicates. Will support
  genre-based exploration in Step 2.
- Both datasets check out as genuine, internally consistent data — no signs
  of synthetic generation or corruption.
- The two datasets are **used in parallel, not merged** — each plays to its
  strengths.

## Next steps

Step 2 will explore both datasets: sound-feature trends by decade from the
audio dataset, and genre/popularity patterns from the metadata dataset,
including genre word clouds.
